## Does the method of calculating the linear regression influence the indicator values?

In order to calculate the two-piecewise linear regression, the *pwlf* Python  is used. 
The underlying implementation this package uses to calculate the breakpoints of the linear regression is differential evolution.
Given that this method is stochastic, we want to ensure that this method does not influence the indicator values, which in turn would influence the analysis for RQ1 and RQ2.
Specifically, we want to determine whether recalculating the indicators many times over can result various values.
To do this, we calculate the coefficient of variation (CV).
To collect data, we did the following. We took our experiment data and recalculated the regression based indicators 30 times over. 
This results in 30 values for each chunk of data used in RQ1 and RQ2.
We calculate the CV for each of these sets of 30 datapoints, and then summarise the outcomes per indicator.


### Data collection (skip over this, its the same as in RQ1)

In [ ]:
import shutil
import os
from behavioural_benchmark.indicators import MemoisedIndicators

current_dir = os.path.dirname(os.getcwd())
fresh_results = os.path.join(current_dir, '../fresh_results')
if not os.path.exists(fresh_results):
    os.makedirs(fresh_results)
result_path = os.path.join(current_dir, "../Results")
if not os.path.exists(result_path):
    os.makedirs(result_path)
    
def get_tags(tag_path, iteration):
    experiment, dimension, population, total_iterations, function_name, metaheuristic, parameters, run = tag_path.split("/")[-8:]
    return experiment, dimension, population, total_iterations, function_name, metaheuristic, control_parameters, run, iteration

def write_results(i: MemoisedIndicators, write_path, write_tags):
    DRoC_A, DRoC_B, ERT_Diversity, Critical_Diversity = i.get_DRoC_A(), i.get_DRoC_B(), i.get_ERT_Diversity(), i.get_Critical_Diversity()
    FRoC_A, FRoC_B, ERT_Fitness, Critical_Fitness = i.get_FRoC_A(), i.get_FRoC_B(), i.get_ERT_Fitness(), i.get_Critical_Fitness()
    SRoC_A, SRoC_B, ERT_Separation, Critical_Separation = i.get_SRoC_A(), i.get_SRoC_B(), i.get_ERT_Separation(), i.get_Critical_Separation()
    MRoC_A, MRoC_B, ERT_Mobility, Critical_Mobility = i.get_MRoC_A(), i.get_MRoC_B(), i.get_ERT_Mobility(), i.get_Critical_Mobility()

    if not os.path.exists(write_path):
        os.makedirs(write_path)
    with open(f"{write_path}/results.csv", 'a+') as f:
        experiment, dimension, pop_size, max_iteration, function, alg, control_params, run, iteration = write_tags
        f.write(
            f"{experiment},{dimension},{pop_size},{max_iteration},{function},{alg},{control_params},{run},{iteration},"
            f"{DRoC_A},{DRoC_B},{ERT_Diversity},{Critical_Diversity},"
            f"{FRoC_A},{FRoC_B},{ERT_Fitness},{Critical_Fitness},"
            f"{SRoC_A},{SRoC_B},{ERT_Separation},{Critical_Separation},"
            f"{MRoC_A},{MRoC_B},{ERT_Mobility},{Critical_Mobility}\n")

def clean(src):
    roots = [dir_root for dir_root, _, _ in os.walk(src)]
    roots.reverse()
    for dir_root in roots:
        if os.path.exists(dir_root) and len(os.listdir(dir_root)) == 0:
            os.rmdir(dir_root)

In [ ]:
expected_experiment_name = "characteristicTest"
for experiment_name in os.listdir(fresh_results):
    if expected_experiment_name not in experiment_name: continue
    for dim in os.listdir(f"{fresh_results}/{experiment_name}"):
        for population_size in os.listdir(f"{fresh_results}/{experiment_name}/{dim}"):
            for max_iter in os.listdir(f"{fresh_results}/{experiment_name}/{dim}/{population_size}"):
                for problem in os.listdir(f"{fresh_results}/{experiment_name}/{dim}/{population_size}/{max_iter}"):
                    problem_root = f"{fresh_results}/{experiment_name}/{dim}/{population_size}/{max_iter}/{problem}"
                    for algorithm in os.listdir(problem_root):
                        for control_parameters in os.listdir(f"{problem_root}/{algorithm}"):
                            root = f"{problem_root}/{algorithm}/{control_parameters}"
                            if not os.path.isdir(root): continue
                            for run in os.listdir(root):
                                input_arg = f"{root}/{run}"
                                for i in range(30):
                                    indicators = MemoisedIndicators(input_arg)
                                    tags = get_tags(indicators.path, i)
                                    write_results(i=indicators, write_path=f"{result_path}/{experiment_name}_sensitivity/", write_tags=tags)
                                shutil.move(
                                    f"{root}/{run}",
                                    f"{root}/{run}".replace("fresh_results", "processed_results")
                                    )
    clean(f"{fresh_results}/{experiment_name}")

# CV calculation

In [1]:
import pandas as pd
import numpy as np

header_tags = ["experiment", "dim", "pop_size", "max_iter", "function", "metaheuristic", "control_params", "run", "iteration"]
indicators = ["DRoC Type A","DRoC Type B","ERT Diversity","Critical Diversity",
              "FRoC Type A","FRoC Type B","ERT Fitness","Critical Fitness",
              "SRoC Type A","SRoC Type B","ERT Separation","Critical Separation",
              "MRoC Type A","MRoC Type B","ERT Mobility","Critical Mobility"]
data = pd.read_csv(f"regression_sensitivity_results.csv", names=header_tags + indicators)

functions = data["function"].unique().tolist()
metaheuristics = data["metaheuristic"].unique().tolist()
indicator_stat_col = [x for xs in [[f"{i}"] * 3 for i in indicators] for x in xs]

In [4]:
results = []
for function in functions:
    for metaheuristic in metaheuristics:
        for run in range(30):
            indicator_results = []
            for indicator in indicators:
                fixed_df = data.loc[(data["function"] == function) & (data["metaheuristic"] == metaheuristic) & (data["run"] == run + 1), [indicator]]
                mean = fixed_df.mean().iloc[0]
                std = fixed_df.std().iloc[0]
                cov = std / mean
                indicator_results.extend([mean, std, cov])
            results.append([function, metaheuristic, run + 1] + indicator_results)

In [5]:
tags = [r"\textit{f}", r"\textit{m}", "run"]
multi_column = pd.MultiIndex.from_arrays([tags + indicator_stat_col, [''] * 3 + ["mean", "std", "cov"] * len(indicators)])
result_df = pd.DataFrame(data=results, columns=multi_column)
result_df.round(2)

\textit{f}     \textit{m} run DRoC Type A           DRoC Type B       \
                                            mean  std  cov        mean  std   
0     Weierstrass  VonNeumannPSO   1       -0.01  0.0 -0.0        -0.0  0.0   
1     Weierstrass  VonNeumannPSO   2       -0.01  0.0 -0.0        -0.0  0.0   
2     Weierstrass  VonNeumannPSO   3       -0.00  0.0 -0.0        -0.0  0.0   
3     Weierstrass  VonNeumannPSO   4       -0.00  0.0 -0.0        -0.0  0.0   
4     Weierstrass  VonNeumannPSO   5       -0.00  0.0 -0.0        -0.0  0.0   
...           ...            ...  ..         ...  ...  ...         ...  ...   
3235        Brown       GBestPSO  26       -0.03  0.0 -0.1        -0.0  0.0   
3236        Brown       GBestPSO  27       -0.03  0.0 -0.0        -0.0  0.0   
3237        Brown       GBestPSO  28       -0.05  0.0 -0.0        -0.0  0.0   
3238        Brown       GBestPSO  29       -0.03  0.0 -0.0        -0.0  0.0   
3239        Brown       GBestPSO  30       -0.02  0.0 -0.0        -0.0  0.0   

           ERT Diversity  ... MRoC Type A MRoC Type B            ERT Mobility  \
       cov          mean  ...         cov        mean  std   cov         mean   
0    -0.09        295.04  ...        -0.0        -0.0  0.0 -0.00        77.51   
1    -0.00        152.31  ...        -0.0        -0.0  0.0 -0.00        87.95   
2    -0.00        154.63  ...        -0.0        -0.0  0.0 -0.00        84.68   
3    -0.01        188.06  ...        -0.0        -0.0  0.0 -0.00        64.75   
4    -0.00        146.62  ...        -0.0        -0.0  0.0 -0.00        30.94   
...    ...           ...  ...         ...         ...  ...   ...          ...   
3235 -0.13         63.08  ...        -0.0        -0.0  0.0 -0.00        32.69   
3236 -0.00         64.50  ...        -0.0        -0.0  0.0 -0.00        35.57   
3237 -0.00         50.31  ...        -0.0        -0.0  0.0 -0.01        36.71   
3238 -0.00        100.43  ...        -0.0        -0.0  0.0 -0.00        48.38   
3239 -0.00        117.31  ...        -0.0        -0.0  0.0 -0.00        34.13   

                Critical Mobility            
       std  cov              mean  std  cov  
0     0.04  0.0              0.02  0.0  0.0  
1     0.03  0.0              0.04  0.0  0.0  
2     0.04  0.0              0.03  0.0  0.0  
3     0.03  0.0              0.03  0.0  0.0  
4     0.01  0.0              0.03  0.0  0.0  
...    ...  ...               ...  ...  ...  
3235  0.00  0.0              0.03  0.0  0.0  
3236  0.00  0.0              0.04  0.0  0.0  
3237  0.00  0.0              0.03  0.0  0.0  
3238  0.00  0.0              0.03  0.0  0.0  
3239  0.00  0.0              0.04  0.0  0.0  

[3240 rows x 51 columns]

In [6]:
cov_df = result_df.loc[:,result_df.columns.get_level_values(1) == 'cov'].abs()

In [7]:
greater_than_1_percent = (cov_df > 0.01).sum() / cov_df.shape[0] * 100
greater_than_5_percent = (cov_df > 0.05).sum() / cov_df.shape[0] * 100
greater_than_10_percent = (cov_df > 0.1).sum() / cov_df.shape[0] * 100
greater_than_20_percent = (cov_df > 0.2).sum() / cov_df.shape[0] * 100

In [9]:
multi_column = pd.MultiIndex.from_arrays(arrays=[["Coefficient of Variance"] * 4, ["\% $>$ 0.01", "\% $>$ 0.05", "\% $>$ 0.1", "\% $>$ 0.2"]])
aggregated = pd.DataFrame(data=np.stack([greater_than_1_percent, greater_than_5_percent, greater_than_10_percent, greater_than_20_percent], axis=1), index=indicators, columns=multi_column)
aggregated.round(2)

Coefficient of Variance                                  
                                \% $>$ 0.01 \% $>$ 0.05 \% $>$ 0.1 \% $>$ 0.2
DRoC Type A                            0.62        0.43       0.37       0.28
DRoC Type B                            1.20        0.59       0.40       0.28
ERT Diversity                          0.65        0.49       0.46       0.40
Critical Diversity                     0.86        0.37       0.19       0.00
FRoC Type A                            1.42        1.17       1.14       0.86
FRoC Type B                            6.79        5.49       4.97       4.35
ERT Fitness                            1.39        1.14       1.11       1.08
Critical Fitness                       1.39        0.99       0.77       0.49
SRoC Type A                            1.91        1.57       1.54       1.27
SRoC Type B                            7.47        5.96       5.62       5.37
ERT Separation                         1.45        1.33       1.30       1.30
Critical Separation                    1.39        0.56       0.28       0.09
MRoC Type A                            8.83        8.52       8.27       7.47
MRoC Type B                            7.47        6.51       6.11       4.88
ERT Mobility                           9.17        8.27       7.69       6.88
Critical Mobility                      7.35        6.17       5.49       3.73

### Summary for the paper

In [10]:
print(aggregated.to_latex(sparsify=True, float_format="%.2f", escape=False, caption="Aggregated summary of coefficient of variance for indicators calculated from a two-piecewise linear regression, when recalculated 30 times. Each column shows the percentage of tests in which a coefficient of variance was returned larger than the given number.", label="tab:CoV"))

\begin{table}
\caption{Aggregated summary of coefficient of variance for indicators calculated from a two-piecewise linear regression, when recalculated 30 times.}
\label{tab:CoV}
\begin{tabular}{lrrrr}
\toprule
 & \multicolumn{4}{r}{Coefficient of Variance} \\
 & \% $>$ 0.01 & \% $>$ 0.05 & \% $>$ 0.1 & \% $>$ 0.2 \\
\midrule
DRoC Type A & 0.62 & 0.43 & 0.37 & 0.28 \\
DRoC Type B & 1.20 & 0.59 & 0.40 & 0.28 \\
ERT Diversity & 0.65 & 0.49 & 0.46 & 0.40 \\
Critical Diversity & 0.86 & 0.37 & 0.19 & 0.00 \\
FRoC Type A & 1.42 & 1.17 & 1.14 & 0.86 \\
FRoC Type B & 6.79 & 5.49 & 4.97 & 4.35 \\
ERT Fitness & 1.39 & 1.14 & 1.11 & 1.08 \\
Critical Fitness & 1.39 & 0.99 & 0.77 & 0.49 \\
SRoC Type A & 1.91 & 1.57 & 1.54 & 1.27 \\
SRoC Type B & 7.47 & 5.96 & 5.62 & 5.37 \\
ERT Separation & 1.45 & 1.33 & 1.30 & 1.30 \\
Critical Separation & 1.39 & 0.56 & 0.28 & 0.09 \\
MRoC Type A & 8.83 & 8.52 & 8.27 & 7.47 \\
MRoC Type B & 7.47 & 6.51 & 6.11 & 4.88 \\
ERT Mobility & 9.17 & 8.27 & 7.69 & 6.88 \\
